In [1]:
# ============================================================
# NOTEBOOK 04: INTEGRACIÓN CON MLFLOW
# Proyecto: Credit Risk Scoring ML
# Autor: Marín Serrato Barrios
# Descripción: Registro de experimentos con MLflow.
#              Versionado y gestión del ciclo de vida
#              del modelo de scoring crediticio.
# ============================================================

import mlflow
import mlflow.lightgbm
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import os
import warnings
warnings.filterwarnings('ignore')

# Ruta base del proyecto
RUTA_PROYECTO = "c:/Users/Marin/Documents/PROYECTO ML_OPS/credit-risk-scoring-ml"
os.chdir(RUTA_PROYECTO)

# Verificar versión de MLflow
print(f"MLflow versión: {mlflow.__version__}")
print(f"Directorio:     {os.getcwd()}")
print("\n✅ Configuración correcta")

MLflow versión: 3.11.1
Directorio:     c:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml

✅ Configuración correcta


In [2]:
# --- CARGA DE DATOS ---

print("Cargando datasets...")

df_lgbm      = pd.read_csv("data/processed/df_lgbm.csv")
df_scorecard = pd.read_csv("data/processed/df_scorecard.csv")

# Separar features y target
X_lgbm     = df_lgbm.drop(columns=['TARGET', 'SK_ID_CURR'])
y          = df_lgbm['TARGET']

X_scorecard = df_scorecard.drop(columns=['TARGET', 'SK_ID_CURR'])
y_score     = df_scorecard['TARGET']

# Imputar NaN en scorecard (igual que en notebook 03)
cols_mediana = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
                'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
                'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE',
                'DAYS_LAST_PHONE_CHANGE', 'AMT_REQ_CREDIT_BUREAU_DAY',
                'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
                'AMT_REQ_CREDIT_BUREAU_YEAR', 'ANTIGUEDAD_LABORAL_AÑOS',
                'EXT_SOURCE_PROMEDIO', 'EXT_SOURCE_MIN',
                'DTI_X_EXT2', 'RIESGO_EDAD_SCORE']

for col in cols_mediana:
    if col in df_scorecard.columns:
        df_scorecard[col] = df_scorecard[col].fillna(
            df_scorecard[col].median())

if 'OWN_CAR_AGE' in df_scorecard.columns:
    df_scorecard['OWN_CAR_AGE'] = df_scorecard['OWN_CAR_AGE'].fillna(0)

X_scorecard = df_scorecard.drop(columns=['TARGET', 'SK_ID_CURR'])

# Split temporal (igual que notebook 03)
idx_ordenado = df_lgbm['SK_ID_CURR'].argsort().values
n_total = len(y)
n_train = int(n_total * 0.70)
n_val   = int(n_total * 0.15)

idx_train = idx_ordenado[:n_train]
idx_val   = idx_ordenado[n_train:n_train + n_val]
idx_test  = idx_ordenado[n_train + n_val:]

X_train = X_lgbm.iloc[idx_train]
X_val   = X_lgbm.iloc[idx_val]
X_test  = X_lgbm.iloc[idx_test]
y_train = y.iloc[idx_train]
y_val   = y.iloc[idx_val]
y_test  = y.iloc[idx_test]

X_train_sc = X_scorecard.iloc[idx_train]
X_val_sc   = X_scorecard.iloc[idx_val]
X_test_sc  = X_scorecard.iloc[idx_test]
y_train_sc = y_score.iloc[idx_train]
y_val_sc   = y_score.iloc[idx_val]
y_test_sc  = y_score.iloc[idx_test]

print(f"✅ Datos cargados: {len(y):,} registros")
print(f"   Train: {len(y_train):,} | Val: {len(y_val):,} | Test: {len(y_test):,}")

Cargando datasets...
✅ Datos cargados: 307,511 registros
   Train: 215,257 | Val: 46,126 | Test: 46,128


In [3]:
# --- CONFIGURACIÓN DE MLFLOW ---

# Definir dónde MLflow guarda sus datos
# Por defecto crea una carpeta mlruns/ en el directorio actual
mlflow.set_tracking_uri("mlruns")

# Crear o conectar al experimento
# Un experimento agrupa todos los runs relacionados
# En nuestro caso: todos los modelos de scoring crediticio
EXPERIMENT_NAME = "scoring_crediticio_home_credit"

mlflow.set_experiment(EXPERIMENT_NAME)

print(f"✅ MLflow configurado")
print(f"   Tracking URI: mlruns/")
print(f"   Experimento:  {EXPERIMENT_NAME}")
print()
print("Cuando entrenes un modelo, MLflow guardará todo en:")
print(f"   {RUTA_PROYECTO}/mlruns/")

2026/04/28 16:49:35 INFO mlflow.tracking.fluent: Experiment with name 'scoring_crediticio_home_credit' does not exist. Creating a new experiment.


✅ MLflow configurado
   Tracking URI: mlruns/
   Experimento:  scoring_crediticio_home_credit

Cuando entrenes un modelo, MLflow guardará todo en:
   c:/Users/Marin/Documents/PROYECTO ML_OPS/credit-risk-scoring-ml/mlruns/


In [4]:
# --- RUN 1: SCORECARD LOGÍSTICO CON MLFLOW ---

# Todo lo que ocurre dentro del bloque "with mlflow.start_run()"
# queda registrado automáticamente en MLflow.

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import mlflow.sklearn

print("Entrenando Scorecard Logístico y registrando en MLflow...")
print()

# Parámetros del modelo
PARAMS_SCORECARD = {
    "C":            0.01,
    "max_iter":     1000,
    "solver":       "lbfgs",
    "class_weight": "balanced",
    "random_state": 42
}

# Iniciamos el run de MLflow
with mlflow.start_run(run_name="scorecard_logistico_v1"):

    # ── PASO 1: Registrar parámetros ──────────────────────
    # Esto guarda QUÉ configuración usaste
    mlflow.log_params(PARAMS_SCORECARD)
    mlflow.log_param("modelo_tipo",   "scorecard_logistico")
    mlflow.log_param("n_features",    X_train_sc.shape[1])
    mlflow.log_param("n_train",       len(y_train_sc))
    mlflow.log_param("split_train",   0.70)
    mlflow.log_param("split_val",     0.15)
    mlflow.log_param("split_test",    0.15)

    # ── PASO 2: Entrenar el modelo ────────────────────────
    pipeline_sc = Pipeline([
        ('scaler',   StandardScaler()),
        ('logistic', LogisticRegression(**PARAMS_SCORECARD))
    ])
    pipeline_sc.fit(X_train_sc, y_train_sc)

    # ── PASO 3: Calcular métricas ─────────────────────────
    def calcular_ks(y_true, y_proba):
        df_ks = pd.DataFrame({'real': y_true, 'proba': y_proba})
        df_ks = df_ks.sort_values('proba', ascending=False).reset_index(drop=True)
        df_ks['cum_malos']  = (df_ks['real']==1).cumsum() / df_ks['real'].sum()
        df_ks['cum_buenos'] = (df_ks['real']==0).cumsum() / (df_ks['real']==0).sum()
        return (df_ks['cum_malos'] - df_ks['cum_buenos']).max()

    proba_train = pipeline_sc.predict_proba(X_train_sc)[:,1]
    proba_val   = pipeline_sc.predict_proba(X_val_sc)[:,1]
    proba_test  = pipeline_sc.predict_proba(X_test_sc)[:,1]

    metricas = {
        "auc_train": roc_auc_score(y_train_sc, proba_train),
        "auc_val":   roc_auc_score(y_val_sc,   proba_val),
        "auc_test":  roc_auc_score(y_test_sc,  proba_test),
        "ks_train":  calcular_ks(y_train_sc, proba_train),
        "ks_val":    calcular_ks(y_val_sc,   proba_val),
        "ks_test":   calcular_ks(y_test_sc,  proba_test),
    }
    metricas["gini_test"] = 2 * metricas["auc_test"] - 1

    # ── PASO 4: Registrar métricas en MLflow ──────────────
    # Esto guarda QUÉ resultados obtuviste
    mlflow.log_metrics(metricas)

    # ── PASO 5: Guardar el modelo en MLflow ───────────────
    # Esto guarda EL MODELO para poder usarlo después
    mlflow.sklearn.log_model(
        pipeline_sc,
        artifact_path="model",
        registered_model_name="scorecard_logistico"
    )

    print("RESULTADOS SCORECARD LOGÍSTICO:")
    print(f"  AUC train: {metricas['auc_train']:.4f}")
    print(f"  AUC val:   {metricas['auc_val']:.4f}")
    print(f"  AUC test:  {metricas['auc_test']:.4f}")
    print(f"  KS test:   {metricas['ks_test']:.4f}")
    print(f"  Gini test: {metricas['gini_test']:.4f}")
    print()
    print("✅ Registrado en MLflow como 'scorecard_logistico'")

Entrenando Scorecard Logístico y registrando en MLflow...



2026/04/28 16:54:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/28 16:54:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RESULTADOS SCORECARD LOGÍSTICO:
  AUC train: 0.7479
  AUC val:   0.7564
  AUC test:  0.7474
  KS test:   0.3657
  Gini test: 0.4949

✅ Registrado en MLflow como 'scorecard_logistico'


Successfully registered model 'scorecard_logistico'.
Created version '1' of model 'scorecard_logistico'.


In [6]:
# --- RUN 2 CORREGIDO: LIGHTGBM ---

print("Reentrenando LightGBM (versión corregida)...")
print()

with mlflow.start_run(run_name="lightgbm_v1_corregido"):

    mlflow.log_params(PARAMS_LGBM)
    mlflow.log_param("modelo_tipo",           "lightgbm")
    mlflow.log_param("n_features",            X_train.shape[1])
    mlflow.log_param("early_stopping_rounds", 50)

    # Callbacks corregidos
    # verbose=True para ver el progreso
    callbacks = [
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=100)
    ]

    model_lgbm = lgb.LGBMClassifier(**PARAMS_LGBM)
    model_lgbm.fit(
        X_train, y_train,
        eval_set=[
            (X_train, y_train),  # ← agregar training también
            (X_val,   y_val)
        ],
        callbacks=callbacks
    )

    print(f"\nMejor iteración: {model_lgbm.best_iteration_}")

    proba_train = model_lgbm.predict_proba(X_train)[:,1]
    proba_val   = model_lgbm.predict_proba(X_val)[:,1]
    proba_test  = model_lgbm.predict_proba(X_test)[:,1]

    metricas_lgbm = {
        "auc_train":      roc_auc_score(y_train, proba_train),
        "auc_val":        roc_auc_score(y_val,   proba_val),
        "auc_test":       roc_auc_score(y_test,  proba_test),
        "ks_train":       calcular_ks(y_train, proba_train),
        "ks_val":         calcular_ks(y_val,   proba_val),
        "ks_test":        calcular_ks(y_test,  proba_test),
        "best_iteration": model_lgbm.best_iteration_,
    }
    metricas_lgbm["gini_test"] = 2 * metricas_lgbm["auc_test"] - 1

    mlflow.log_metrics(metricas_lgbm)

    mlflow.lightgbm.log_model(
        model_lgbm,
        name="model",
        registered_model_name="lightgbm_scoring"
    )

    mlflow.set_tag("champion",    "True")
    mlflow.set_tag("dataset",     "home_credit_sin_bureau")
    mlflow.set_tag("responsable", "Marin Serrato")

    print("\nRESULTADOS LIGHTGBM:")
    print(f"  AUC train: {metricas_lgbm['auc_train']:.4f}")
    print(f"  AUC val:   {metricas_lgbm['auc_val']:.4f}")
    print(f"  AUC test:  {metricas_lgbm['auc_test']:.4f}")
    print(f"  KS test:   {metricas_lgbm['ks_test']:.4f}")
    print(f"  Gini test: {metricas_lgbm['gini_test']:.4f}")
    print(f"  Iteraciones: {metricas_lgbm['best_iteration']:.0f}")
    print()
    print("✅ Registrado en MLflow como 'lightgbm_scoring' versión 2")

Reentrenando LightGBM (versión corregida)...

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	training's binary_logloss: 0.278432	valid_1's binary_logloss: 0.275165

Mejor iteración: 1


2026/04/28 17:02:07 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



RESULTADOS LIGHTGBM:
  AUC train: 0.7038
  AUC val:   0.7113
  AUC test:  0.7052
  KS test:   0.3187
  Gini test: 0.4103
  Iteraciones: 1

✅ Registrado en MLflow como 'lightgbm_scoring' versión 2


Registered model 'lightgbm_scoring' already exists. Creating a new version of this model...
Created version '2' of model 'lightgbm_scoring'.


In [7]:
# --- RUN 3: LIGHTGBM SIN EARLY STOPPING (diagnóstico) ---

print("Diagnóstico: LightGBM sin early stopping...")
print()

# Parámetros simplificados para diagnóstico
params_simple = {
    'objective':         'binary',
    'n_estimators':      568,    # el número óptimo que encontramos antes
    'max_depth':         4,
    'num_leaves':        31,
    'learning_rate':     0.05,
    'min_child_samples': 50,
    'reg_alpha':         0.1,
    'reg_lambda':        1.0,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
    'scale_pos_weight':  11.4,
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
}

with mlflow.start_run(run_name="lightgbm_v1_final"):

    mlflow.log_params(params_simple)
    mlflow.log_param("modelo_tipo", "lightgbm")
    mlflow.log_param("n_features",  X_train.shape[1])
    mlflow.log_param("metodo",      "n_estimators_fijo_568")

    # Entrenar sin early stopping
    # Usamos directamente el número óptimo de árboles
    # que encontramos en el notebook 03
    model_lgbm = lgb.LGBMClassifier(**params_simple)
    model_lgbm.fit(X_train, y_train)

    print(f"Entrenamiento completado: {params_simple['n_estimators']} árboles")

    proba_train = model_lgbm.predict_proba(X_train)[:,1]
    proba_val   = model_lgbm.predict_proba(X_val)[:,1]
    proba_test  = model_lgbm.predict_proba(X_test)[:,1]

    metricas_lgbm = {
        "auc_train":  roc_auc_score(y_train, proba_train),
        "auc_val":    roc_auc_score(y_val,   proba_val),
        "auc_test":   roc_auc_score(y_test,  proba_test),
        "ks_train":   calcular_ks(y_train, proba_train),
        "ks_val":     calcular_ks(y_val,   proba_val),
        "ks_test":    calcular_ks(y_test,  proba_test),
    }
    metricas_lgbm["gini_test"] = 2 * metricas_lgbm["auc_test"] - 1

    mlflow.log_metrics(metricas_lgbm)

    mlflow.lightgbm.log_model(
        model_lgbm,
        name="model",
        registered_model_name="lightgbm_scoring"
    )

    mlflow.set_tag("champion",    "True")
    mlflow.set_tag("dataset",     "home_credit_sin_bureau")
    mlflow.set_tag("responsable", "Marin Serrato")

    print("\nRESULTADOS LIGHTGBM:")
    print(f"  AUC train: {metricas_lgbm['auc_train']:.4f}")
    print(f"  AUC val:   {metricas_lgbm['auc_val']:.4f}")
    print(f"  AUC test:  {metricas_lgbm['auc_test']:.4f}")
    print(f"  KS test:   {metricas_lgbm['ks_test']:.4f}")
    print(f"  Gini test: {metricas_lgbm['gini_test']:.4f}")
    print()
    print("✅ Registrado en MLflow como 'lightgbm_scoring' versión 3")

Diagnóstico: LightGBM sin early stopping...

Entrenamiento completado: 568 árboles


2026/04/28 17:08:17 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



RESULTADOS LIGHTGBM:
  AUC train: 0.8056
  AUC val:   0.7731
  AUC test:  0.7679
  KS test:   0.4000
  Gini test: 0.5359

✅ Registrado en MLflow como 'lightgbm_scoring' versión 3


Registered model 'lightgbm_scoring' already exists. Creating a new version of this model...
Created version '3' of model 'lightgbm_scoring'.


In [8]:
# --- PASO: PROMOVER LIGHTGBM A PRODUCCIÓN EN EL REGISTRY ---

# El Model Registry tiene tres etapas:
#
# None     → modelo recién registrado, sin estado
# Staging  → modelo en pruebas, validado técnicamente
# Production → modelo aprobado para producción
# Archived → modelo retirado
#
# Vamos a promover lightgbm_scoring versión 3 a Production

from mlflow import MlflowClient

client = MlflowClient()

# Promover a Production
client.set_registered_model_alias(
    name="lightgbm_scoring",
    alias="production",
    version="3"
)

# Agregar descripción al modelo
client.update_registered_model(
    name="lightgbm_scoring",
    description="Modelo de scoring crediticio para originación. "
                "Entrenado con Home Credit Default Risk dataset. "
                "AUC=0.768, KS=0.400. "
                "Supera al scorecard logístico en todas las métricas."
)

# Agregar descripción a esta versión específica
client.update_model_version(
    name="lightgbm_scoring",
    version="3",
    description="Versión final con 568 árboles. "
                "Sin bureau.csv. "
                "Umbral óptimo: 0.59. "
                "Mora en aprobados: 4.5% vs 8.1% original."
)

print("✅ lightgbm_scoring v3 promovido a Production")
print()
print("Estado en el Registry:")
model = client.get_registered_model("lightgbm_scoring")
print(f"  Modelo: {model.name}")
print(f"  Descripción: {model.description}")
print()
for version in client.search_model_versions(f"name='lightgbm_scoring'"):
    alias = "production" if version.version == "3" else "-"
    print(f"  Versión {version.version}: {version.description[:50]}... alias={alias}")

✅ lightgbm_scoring v3 promovido a Production

Estado en el Registry:
  Modelo: lightgbm_scoring
  Descripción: Modelo de scoring crediticio para originación. Entrenado con Home Credit Default Risk dataset. AUC=0.768, KS=0.400. Supera al scorecard logístico en todas las métricas.

  Versión 3: Versión final con 568 árboles. Sin bureau.csv. Umb... alias=-


TypeError: 'NoneType' object is not subscriptable

In [9]:
# --- VERIFICACIÓN CORREGIDA ---

print("Estado en el Registry:")
print()

for version in client.search_model_versions("name='lightgbm_scoring'"):
    # Manejar versiones sin descripción
    desc = version.description[:50] if version.description else "Sin descripción"
    
    # Verificar alias
    aliases = version.aliases if hasattr(version, 'aliases') else []
    alias_str = "production" if "production" in aliases else "-"
    
    print(f"  Versión {version.version}:")
    print(f"    Descripción: {desc}")
    print(f"    Alias:       {alias_str}")
    print()

Estado en el Registry:

  Versión 3:
    Descripción: Versión final con 568 árboles. Sin bureau.csv. Umb
    Alias:       production

  Versión 2:
    Descripción: Sin descripción
    Alias:       -

  Versión 1:
    Descripción: Sin descripción
    Alias:       -



In [10]:
# --- CARGAR MODELO DESDE MLFLOW Y PREDECIR ---

# Esta es la prueba más importante:
# ¿Podemos recuperar el modelo guardado
# y usarlo para predecir sin reentrenarlo?

print("CARGANDO MODELO DESDE MLFLOW...")
print("="*60)

# Cargar el modelo usando su alias "production"
# No necesitamos saber el número de versión
# Solo el alias que le asignamos
modelo_produccion = mlflow.lightgbm.load_model(
    "models:/lightgbm_scoring@production"
)

print("✅ Modelo cargado desde MLflow")
print(f"   Tipo: {type(modelo_produccion)}")
print()

# Hacer predicciones con los primeros 5 clientes del test
muestra = X_test.head(5)
probabilidades = modelo_produccion.predict_proba(muestra)[:,1]

print("PREDICCIONES PARA 5 CLIENTES DE PRUEBA:")
print("-"*60)
for i, (prob, real) in enumerate(
        zip(probabilidades, y_test.head(5))):
    decision = "RECHAZAR" if prob >= 0.59 else "APROBAR"
    print(f"  Cliente {i+1}: P(mora)={prob:.4f} "
          f"→ {decision} "
          f"(real: {'MALO' if real==1 else 'BUENO'})")

print()
print("✅ El modelo funciona correctamente desde MLflow")
print("   Sin reentrenar, sin cargar el notebook de modelado")
print("   Solo con el alias 'production'")

CARGANDO MODELO DESDE MLFLOW...
✅ Modelo cargado desde MLflow
   Tipo: <class 'lightgbm.sklearn.LGBMClassifier'>

PREDICCIONES PARA 5 CLIENTES DE PRUEBA:
------------------------------------------------------------
  Cliente 1: P(mora)=0.2459 → APROBAR (real: BUENO)
  Cliente 2: P(mora)=0.3992 → APROBAR (real: BUENO)
  Cliente 3: P(mora)=0.3691 → APROBAR (real: BUENO)
  Cliente 4: P(mora)=0.7440 → RECHAZAR (real: BUENO)
  Cliente 5: P(mora)=0.1670 → APROBAR (real: BUENO)

✅ El modelo funciona correctamente desde MLflow
   Sin reentrenar, sin cargar el notebook de modelado
   Solo con el alias 'production'


In [11]:
# --- SIMULACIÓN DE SCORING EN PRODUCCIÓN ---

# Esta celda simula exactamente lo que ocurriría
# cuando el sistema de originación del banco
# envía una solicitud al modelo.
#
# En producción real esto sería una API REST.
# Por ahora lo simulamos como función Python.

print("SIMULACIÓN DE SCORING EN PRODUCCIÓN")
print("="*60)

def scoring_credito(features_cliente: dict) -> dict:
    """
    Función que simula el endpoint de scoring.
    
    Recibe: diccionario con las features del cliente
    Devuelve: decisión de crédito con explicación
    
    En producción real esta función estaría
    dentro de una API REST en Azure ML.
    """
    
    # Cargar modelo desde MLflow (en producción
    # esto se hace UNA SOLA VEZ al iniciar el servicio)
    modelo = mlflow.lightgbm.load_model(
        "models:/lightgbm_scoring@production"
    )
    
    # Convertir el diccionario a DataFrame
    # (LightGBM necesita DataFrame, no diccionario)
    df_cliente = pd.DataFrame([features_cliente])
    
    # Asegurar que las columnas están en el orden correcto
    columnas_modelo = X_test.columns.tolist()
    for col in columnas_modelo:
        if col not in df_cliente.columns:
            df_cliente[col] = 0  # valor por defecto
    df_cliente = df_cliente[columnas_modelo]
    
    # Predecir probabilidad de mora
    prob_mora = modelo.predict_proba(df_cliente)[0, 1]
    
    # Aplicar umbral de decisión
    UMBRAL = 0.59
    decision = "RECHAZADO" if prob_mora >= UMBRAL else "APROBADO"
    
    # Nivel de riesgo
    if prob_mora < 0.20:
        nivel_riesgo = "BAJO"
    elif prob_mora < 0.40:
        nivel_riesgo = "MEDIO"
    elif prob_mora < 0.59:
        nivel_riesgo = "ALTO"
    else:
        nivel_riesgo = "MUY ALTO"
    
    return {
        "probabilidad_mora":  round(float(prob_mora), 4),
        "decision":           decision,
        "umbral_aplicado":    UMBRAL,
        "nivel_riesgo":       nivel_riesgo,
        "modelo_version":     "lightgbm_scoring@production",
    }

# ── PRUEBA CON TRES PERFILES DE CLIENTES ──────────────────

# Tomamos tres clientes reales del conjunto de test
# con diferentes niveles de riesgo

print("PRUEBA CON CLIENTES REALES DEL TEST SET:")
print("-"*60)

for i, idx in enumerate([0, 100, 500]):
    features = X_test.iloc[idx].to_dict()
    resultado = scoring_credito(features)
    real = "MALO" if y_test.iloc[idx] == 1 else "BUENO"
    
    print(f"\nCliente {i+1} (real: {real}):")
    print(f"  P(mora):      {resultado['probabilidad_mora']:.4f}")
    print(f"  Nivel riesgo: {resultado['nivel_riesgo']}")
    print(f"  Decisión:     {resultado['decision']}")
    print(f"  Modelo:       {resultado['modelo_version']}")

SIMULACIÓN DE SCORING EN PRODUCCIÓN
PRUEBA CON CLIENTES REALES DEL TEST SET:
------------------------------------------------------------

Cliente 1 (real: BUENO):
  P(mora):      0.2459
  Nivel riesgo: MEDIO
  Decisión:     APROBADO
  Modelo:       lightgbm_scoring@production

Cliente 2 (real: BUENO):
  P(mora):      0.5247
  Nivel riesgo: ALTO
  Decisión:     APROBADO
  Modelo:       lightgbm_scoring@production

Cliente 3 (real: BUENO):
  P(mora):      0.2763
  Nivel riesgo: MEDIO
  Decisión:     APROBADO
  Modelo:       lightgbm_scoring@production
